In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 27. Flash Attentionテキスト テキスト テキスト

> **学習目標**
> - Flash Attentionテキスト テキスト テキスト(GPU SRAM vs HBM)テキスト テキスト テキスト テキスト
> - IO テキスト度 テキスト テキスト(tiling)テキスト テキスト
> - テキスト テキスト テキスト テキスト テキスト

## 27.1 GPU テキスト テキスト

GPUテキスト テキスト テキスト テキスト:
- **SRAM** (テキスト, テキスト テキスト, テキスト ~20MB)
- **HBM** (High Bandwidth Memory, テキスト "GPU テキスト"テキスト テキスト, ~40-80GB)
- **DRAM** (CPU テキスト, テキスト, テキスト)

HBM ↔ SRAM テキスト テキスト. Attentionテキスト $n \times n$ 行列テキスト HBMテキスト テキスト テキスト → テキスト.

## 27.2 テキスト Attentionテキスト テキスト テキスト

$$S = QK^\top \in \mathbb{R}^{n \times n}$$  (HBMテキスト テキスト)
$$P = \mathrm{softmax}(S) \in \mathbb{R}^{n \times n}$$  (HBM テキスト テキスト)
$$O = PV \in \mathbb{R}^{n \times d}$$  (HBM テキスト テキスト)

- HBM テキスト: $O(n^2)$ (テキスト)
- SRAM テキスト: テキスト テキスト

## 27.3 Flash Attention — テキスト Online Softmax

**テキスト テキスト**: $n \times n$ 行列テキスト テキスト テキスト テキスト, テキスト テキスト SRAMテキスト テキスト.

### Online Softmax
テキスト テキスト テキスト 計算テキスト テキスト テキスト, テキスト $\max$テキスト $\sum$テキスト テキスト 計算:
$$m_i^{(j)} = \max(m_i^{(j-1)}, \max_j S_{ij})$$
$$\ell_i^{(j)} = e^{m_i^{(j-1)} - m_i^{(j)}} \ell_i^{(j-1)} + \sum_j e^{S_{ij} - m_i^{(j)}}$$

### Tiling
$Q, K, V$テキスト テキスト テキスト SRAMテキスト テキスト, テキスト Attention 計算 テキスト 結果テキスト HBMテキスト テキスト.

**IO テキスト度**:
- テキスト: $O(n^2)$ HBM テキスト
- Flash: $O(n^2 d / M)$ where $M$ = SRAM テキスト → テキスト テキスト

### バックプロパゲーション テキスト計算
バックプロパゲーション テキスト $S, P$テキスト テキスト テキスト テキスト計算 (テキスト $O(n)$ vs $O(n^2)$).


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# テキスト Attention
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    S = Q @ K.transpose(-1, -2) / (d_k ** 0.5)  # HBM テキスト
    P = F.softmax(S, dim=-1)  # HBM テキスト/テキスト
    O = P @ V  # HBM テキスト/テキスト
    return O

# Flash Attention テキスト (テキスト, online softmax)
def flash_attention_naive(Q, K, V, block_size=64):
    """Flash attentionテキスト テキスト テキスト テキスト テキスト."""
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)
    # Qテキスト Blockテキスト, K/Vテキスト テキスト (テキスト Flashテキスト テキスト テキスト Block)
    for i in range(0, N, block_size):
        Q_block = Q[:, :, i:i+block_size, :]  # (B, H, bs, D)
        S_block = Q_block @ K.transpose(-1, -2) / (D ** 0.5)  # (B, H, bs, N)
        P_block = F.softmax(S_block, dim=-1)
        O[:, :, i:i+block_size, :] = P_block @ V
    return O

# テキスト テキスト
torch.manual_seed(0)
B, H, N, D = 1, 2, 128, 32
Q = torch.randn(B, H, N, D)
K = torch.randn(B, H, N, D)
V = torch.randn(B, H, N, D)

O_std = standard_attention(Q, K, V)
O_flash = flash_attention_naive(Q, K, V, block_size=32)
print(f"テキスト vs Flash テキスト 誤差: {(O_std - O_flash).abs().max():.2e}")
print("=> Resultテキスト テキスト. Flashテキスト Memory Efficiencyテキスト テキストLine.")


## 27.4 PyTorch SDPA テキスト

PyTorchテキスト `scaled_dot_product_attention`テキスト テキスト テキスト テキスト テキスト:
- `flash`: Flash Attention v2 (GPU)
- `mem_efficient`: Memory-Efficient Attention (GPU/CPU)
- `math`: テキスト テキスト


In [ ]:
# SDPA テキスト 比較
import time

# テキスト テキスト テキスト テキスト テキスト
def bench_attention(n, d=64, n_heads=8, device='cpu'):
    """Attention Timeテキスト Memory Measurement."""
    Q = torch.randn(1, n_heads, n, d, device=device)
    K = torch.randn(1, n_heads, n, d, device=device)
    V = torch.randn(1, n_heads, n, d, device=device)

    # Standard Implementation
    def std_attn():
        scores = Q @ K.transpose(-1, -2) / (d ** 0.5)
        weights = F.softmax(scores, dim=-1)
        return weights @ V

    # SDPA
    def sdpa_attn():
        return F.scaled_dot_product_attention(Q, K, V)

    # Time Measurement
    for _ in range(2): std_attn()  # warmup
    t0 = time.perf_counter()
    for _ in range(3):
        std_attn()
    t_std = (time.perf_counter() - t0) / 3 * 1000

    for _ in range(2): sdpa_attn()
    t0 = time.perf_counter()
    for _ in range(3):
        sdpa_attn()
    t_sdpa = (time.perf_counter() - t0) / 3 * 1000

    return t_std, t_sdpa

print(f"{'n':>6} | {'Standard (ms)':>15} | {'SDPA (ms)':>12} | {'Speedup':>10}")
print("-" * 55)
for n in [256, 512, 1024, 2048]:
    t_std, t_sdpa = bench_attention(n, device='cpu')
    print(f"{n:>6} | {t_std:>15.3f} | {t_sdpa:>12.3f} | {t_std/t_sdpa:>9.2f}x")


## 27.5 テキスト テキスト テキスト

テキスト Attentionテキスト $n \times n$ Attention 行列 テキスト → $O(n^2)$ テキスト.

Flash Attentionテキスト Attention 行列 テキスト テキスト テキスト → $O(n)$ テキスト.

テキスト: $n = 8192, h = 32, d_k = 128, B = 1$:
- テキスト: $B \cdot h \cdot n^2 \cdot 4 = 8$ GB (FP32)
- Flash: $O(n)$ → テキスト 100MB


In [ ]:
# テキスト テキスト
def attention_memory_gb(n, n_heads, d_k, batch=1, dtype_bytes=4):
    """Standard Attentionテキスト Attention Matrix Memory."""
    return batch * n_heads * n * n * dtype_bytes / (1024**3)

print(f"Attention Matrix Memory (FP32):")
print(f"{'n':>6} | {'Memory (GB)':>12} | {'Flash (Estimation)':>14}")
print("-" * 40)
for n in [1024, 2048, 4096, 8192, 16384]:
    mem = attention_memory_gb(n, n_heads=32, d_k=128)
    flash_mem = n * 32 * 128 * 4 / (1024**3)  # テキスト O(n)
    print(f"{n:>6} | {mem:>12.3f} | {flash_mem:>14.4f}")
print("\n=> n=8192テキスト Standardテキスト 8GB, Flashテキスト テキスト MB. Difference テキスト度テキスト.")


## 27.6 Flash Attention v2/v3

- **v1** (Dao et al., 2022): テキスト, online softmax
- **v2** (2023): テキスト テキスト テキスト, バックプロパゲーション テキスト
- **v3** (2024): Hopper テキスト (H100) テキスト, テキスト テキスト

## 27.7 Ring Attention — テキスト GPU テキスト

$Q, K, V$テキスト GPUテキスト テキスト, GPU テキスト テキスト テキスト Attention.
- テキスト GPU テキスト テキスト テキスト
- 1M+ テキスト テキスト テキスト

## 27.8 要点

| テキスト | IO テキスト度 | テキスト | 速度 |
|---|---|---|---|
| テキスト Attention | $O(n^2)$ | $O(n^2)$ | テキスト |
| Flash Attention v2 | $O(n^2 d/M)$ | $O(n)$ | テキスト |
| Flash Attention v3 | テキスト | $O(n)$ | H100テキスト テキスト |
| Ring Attention | テキスト | $O(n/k)$ (k GPU) | テキスト テキスト テキスト |

## 演習問題

1. PyTorch SDPAテキスト テキスト `math`, `mem_efficient`テキスト テキスト テキスト 比較テキスト.
2. シーケンス長 1024, 4096, 16384テキスト テキスト vs SDPA 時間テキスト テキスト 比較テキスト.
3. Online Softmaxテキスト テキスト テキスト テキスト テキスト 結果テキスト テキスト テキスト.
4. Flash Attentionテキスト バックプロパゲーション テキスト テキスト テキスト テキスト.
5. Ring Attentionテキスト テキスト GPUテキスト テキスト テキスト テキスト.

> 解答: `solutions/ch27_solutions.ipynb`
